# Notebook 21
## Final Comparison: All Paradigms (DNA + Protein)

This notebook consolidates results from all completed notebooks into clean
comparison tables and publication-quality figures.

### DNA paradigms compared
- Baseline ML (k-mer / GC / CpG features)
- Sequence CNN
- Hybrid (CNN + features)
- DNABERT-2 (117M, BPE)
- Nucleotide Transformer (500M, human-ref)

### Protein paradigms compared
- Baseline ML (amino acid composition + physicochemical features)
- Sequence CNN
- Hybrid (CNN + features)
- ESM-2 (35M, UR50D)
- ProtBERT (420M, UniRef100)

### Outputs
- `reports/dna_final_comparison.csv`
- `reports/protein_final_comparison.csv`
- `reports/figures/final/dna_paradigm_comparison.png`
- `reports/figures/final/protein_paradigm_comparison.png`
- `reports/final_summary.json`

## 1) Imports

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
print('imports OK')


imports OK


## 2) Paths

In [2]:
ROOT    = Path.cwd().parents[0]
REPORTS = ROOT / 'reports'
FIGURES = REPORTS / 'figures' / 'final'
MODELS  = ROOT / 'models'
FIGURES.mkdir(parents=True, exist_ok=True)
print('Reports dir:', REPORTS)


Reports dir: /home/dpratapa/Capstone/reports


## 3) Assemble DNA results

Loads from each paradigm's saved CSV. The hybrid result is loaded from its
JSON summary since notebook 08 did not save a per-model CSV.
Each row is tagged with its paradigm for the combined table.

In [3]:
dna_frames = []

# ---- Baseline (notebook 06) ----
p = REPORTS / 'dna_baseline_test_results_len200.csv'
if p.exists():
    df = pd.read_csv(p)
    df.insert(0, 'paradigm', 'Baseline')
    dna_frames.append(df)
    print('Loaded baseline:', df.shape)
else:
    print('NOT FOUND:', p.name)

# ---- CNN (notebook 07) ----
p = REPORTS / 'dna_seq_cnn_test_results_len200.csv'
if p.exists():
    df = pd.read_csv(p)
    df.insert(0, 'paradigm', 'CNN')
    dna_frames.append(df)
    print('Loaded CNN:', df.shape)
else:
    print('NOT FOUND:', p.name)

# ---- Hybrid (notebook 08) -- JSON only ----
p = REPORTS / 'dna_hybrid_summary_len200.json'
if p.exists():
    with open(p) as f:
        h = json.load(f)
    hybrid_row = pd.DataFrame([{
        'paradigm': 'Hybrid',
        'model': 'cnn+features',
        'accuracy':  h.get('accuracy'),
        'precision': h.get('precision'),
        'recall':    h.get('recall'),
        'f1':        h.get('f1'),
        'roc_auc':   h.get('roc_auc'),
        'pr_auc':    h.get('pr_auc'),
    }])
    dna_frames.append(hybrid_row)
    print('Loaded Hybrid (from JSON)')
else:
    print('NOT FOUND:', p.name)

# ---- DNABERT-2 (notebook 16) ----
p = REPORTS / 'dna_dnabert2_test_results.csv'
if p.exists():
    df = pd.read_csv(p)
    df.insert(0, 'paradigm', 'DNABERT-2')
    dna_frames.append(df)
    print('Loaded DNABERT-2:', df.shape)
else:
    print('NOT FOUND:', p.name)

# ---- Nucleotide Transformer (notebook 18) ----
p = REPORTS / 'dna_nt_test_results.csv'
if p.exists():
    df = pd.read_csv(p)
    df.insert(0, 'paradigm', 'NT-500M')
    dna_frames.append(df)
    print('Loaded NT-500M:', df.shape)
else:
    print('NOT FOUND:', p.name)

dna_all = pd.concat(dna_frames, ignore_index=True)
print(f'\nTotal DNA rows: {len(dna_all)}')


Loaded baseline: (5, 9)
Loaded CNN: (1, 9)
Loaded Hybrid (from JSON)
Loaded DNABERT-2: (4, 8)
Loaded NT-500M: (4, 8)

Total DNA rows: 15


## 4) DNA best-per-paradigm table

Select the best model per paradigm by ROC-AUC, then display a clean summary.

In [4]:
# Best model per paradigm by ROC-AUC
dna_best = (
    dna_all
    .sort_values('roc_auc', ascending=False)
    .groupby('paradigm', sort=False)
    .first()
    .reset_index()
)

# Order paradigms from simplest to most complex
paradigm_order = ['Baseline', 'CNN', 'Hybrid', 'DNABERT-2', 'NT-500M']
dna_best['paradigm'] = pd.Categorical(
    dna_best['paradigm'], categories=paradigm_order, ordered=True)
dna_best = dna_best.sort_values('paradigm').reset_index(drop=True)

cols = [c for c in ['paradigm','model','accuracy','f1','roc_auc','pr_auc']
        if c in dna_best.columns]
print('DNA best-per-paradigm (sorted by complexity):')
print(dna_best[cols].to_string(index=False))

dna_best[cols].to_csv(REPORTS / 'dna_final_comparison.csv', index=False)
print('\nSaved: reports/dna_final_comparison.csv')


DNA best-per-paradigm (sorted by complexity):
 paradigm         model  accuracy       f1  roc_auc   pr_auc
 Baseline random_forest   0.74750 0.743003 0.837384 0.847125
      CNN       seq_cnn   0.75000 0.748744 0.826375 0.837063
   Hybrid  cnn+features   0.69750 0.740343 0.831688 0.846602
DNABERT-2       xgboost   0.75625 0.755332 0.846250 0.853206
  NT-500M       xgboost   0.76875 0.767296 0.850700 0.854365

Saved: reports/dna_final_comparison.csv


## 5) Assemble protein results

The ESM-2 results were saved to `models/protein/esm2/esm2_results_summary.json`
(not in reports/) by notebook 14. All other paradigms have CSVs in reports/.

In [5]:
protein_frames = []

# ---- Baseline (notebook 10) ----
p = REPORTS / 'protein_baseline_test_results_top10_per400.csv'
if p.exists():
    df = pd.read_csv(p)
    df.insert(0, 'paradigm', 'Baseline')
    protein_frames.append(df)
    print('Loaded Baseline:', df.shape)
else:
    print('NOT FOUND:', p.name)

# ---- CNN (notebook 11) ----
p = REPORTS / 'protein_seq_cnn_test_results_top10_per400.csv'
if p.exists():
    df = pd.read_csv(p)
    df.insert(0, 'paradigm', 'CNN')
    protein_frames.append(df)
    print('Loaded CNN:', df.shape)
else:
    print('NOT FOUND:', p.name)

# ---- Hybrid (notebook 12) ----
p = REPORTS / 'protein_hybrid_test_results_top10_per400.csv'
if p.exists():
    df = pd.read_csv(p)
    df.insert(0, 'paradigm', 'Hybrid')
    protein_frames.append(df)
    print('Loaded Hybrid:', df.shape)
else:
    # Try JSON fallback
    pj = REPORTS / 'protein_hybrid_summary_top10_per400.json'
    if pj.exists():
        with open(pj) as f:
            h = json.load(f)
        tr = h.get('test_results', h)  # handle flat or nested
        if isinstance(tr, list):
            df = pd.DataFrame(tr)
        else:
            df = pd.DataFrame([tr])
        df.insert(0, 'paradigm', 'Hybrid')
        protein_frames.append(df)
        print('Loaded Hybrid (from JSON)')
    else:
        print('NOT FOUND: protein_hybrid_*')

# ---- ESM-2 (notebook 14) -- results in models/ not reports/ ----
esm2_json = MODELS / 'protein' / 'esm2' / 'esm2_results_summary.json'
if esm2_json.exists():
    with open(esm2_json) as f:
        esm2_data = json.load(f)
    tr = esm2_data.get('test_results', [])
    df = pd.DataFrame(tr)
    df.insert(0, 'paradigm', 'ESM-2')
    protein_frames.append(df)
    print('Loaded ESM-2 (from JSON):', df.shape)
else:
    print('NOT FOUND:', esm2_json)

# ---- ProtBERT (notebook 20) ----
p = REPORTS / 'protein_protbert_test_results.csv'
if p.exists():
    df = pd.read_csv(p)
    df.insert(0, 'paradigm', 'ProtBERT')
    protein_frames.append(df)
    print('Loaded ProtBERT:', df.shape)
else:
    print('NOT FOUND:', p.name)

protein_all = pd.concat(protein_frames, ignore_index=True)
print(f'\nTotal protein rows: {len(protein_all)}')
print(protein_all.columns.tolist())


Loaded Baseline: (5, 7)
Loaded CNN: (1, 9)
Loaded Hybrid: (1, 9)
Loaded ESM-2 (from JSON): (4, 5)
Loaded ProtBERT: (4, 7)

Total protein rows: 15
['paradigm', 'accuracy', 'precision_w', 'recall_w', 'f1_w', 'n_test', 'model', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'n_classes', 'max_len', 'f1_macro', 'precision_macro', 'recall_macro', 'roc_auc_ovr']


## 6) Protein best-per-paradigm table

Normalise metric columns across paradigms (some use `f1`, others `f1_macro`)
and select best model per paradigm by accuracy.

In [6]:
# Normalise: create unified accuracy / f1_macro columns
if 'f1_macro' not in protein_all.columns and 'f1' in protein_all.columns:
    protein_all['f1_macro'] = protein_all['f1']
elif 'f1_macro' not in protein_all.columns:
    protein_all['f1_macro'] = np.nan

if 'f1' in protein_all.columns and 'f1_macro' in protein_all.columns:
    protein_all['f1_macro'] = protein_all['f1_macro'].fillna(protein_all['f1'])

protein_best = (
    protein_all
    .sort_values('accuracy', ascending=False)
    .groupby('paradigm', sort=False)
    .first()
    .reset_index()
)

paradigm_order = ['Baseline', 'CNN', 'Hybrid', 'ESM-2', 'ProtBERT']
protein_best['paradigm'] = pd.Categorical(
    protein_best['paradigm'], categories=paradigm_order, ordered=True)
protein_best = protein_best.sort_values('paradigm').reset_index(drop=True)

cols = [c for c in ['paradigm','model','accuracy','f1_macro']
        if c in protein_best.columns]
print('Protein best-per-paradigm (sorted by complexity):')
print(protein_best[cols].to_string(index=False))

protein_best[cols].to_csv(REPORTS / 'protein_final_comparison.csv', index=False)
print('\nSaved: reports/protein_final_comparison.csv')


Protein best-per-paradigm (sorted by complexity):
paradigm                 model  accuracy  f1_macro
Baseline               xgboost  0.923747       NaN
     CNN          sequence_cnn  0.924419       NaN
  Hybrid      hybrid_cnn_feats  0.956395       NaN
   ESM-2            linear_svm  0.986928  0.989041
ProtBERT linear_svm_calibrated  0.991285  0.992410

Saved: reports/protein_final_comparison.csv


## 7) DNA comparison bar chart

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

paradigms = dna_best['paradigm'].tolist()
x = np.arange(len(paradigms))
width = 0.35

ax = axes[0]
bars1 = ax.bar(x - width/2,
               dna_best['accuracy'].values,
               width, label='Accuracy', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2,
               dna_best['roc_auc'].values,
               width, label='ROC-AUC', color='coral', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(paradigms, rotation=20, ha='right', fontsize=9)
ax.set_ylim(0.65, 0.92)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_title('DNA: Promoter vs Non-Promoter\n(best model per paradigm)', fontsize=10)
ax.set_ylabel('Score')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Annotate bars with values
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
            f'{h:.3f}', ha='center', va='bottom', fontsize=6.5)

# F1 line overlay
if 'f1' in dna_best.columns:
    ax2 = ax.twinx()
    ax2.plot(x, dna_best['f1'].values, 'g^--', lw=1.5,
             markersize=6, label='F1', alpha=0.8)
    ax2.set_ylim(0.65, 0.92)
    ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax2.set_ylabel('F1', color='green')
    ax2.legend(loc='lower right', fontsize=8)

# --- Protein panel ---
ax = axes[1]
paradigms_p = protein_best['paradigm'].tolist()
xp = np.arange(len(paradigms_p))

bars1p = ax.bar(xp,
                protein_best['accuracy'].values,
                width*1.4, label='Accuracy', color='steelblue', alpha=0.85)
ax.set_xticks(xp)
ax.set_xticklabels(paradigms_p, rotation=20, ha='right', fontsize=9)
ax.set_ylim(0.60, 1.05)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_title('Protein: Pfam Family Classification (10-class)\n(best model per paradigm)', fontsize=10)
ax.set_ylabel('Accuracy')
ax.grid(axis='y', alpha=0.3)

for bar in bars1p:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.004,
            f'{h:.3f}', ha='center', va='bottom', fontsize=8)

if 'f1_macro' in protein_best.columns:
    ax2p = ax.twinx()
    ax2p.plot(xp, protein_best['f1_macro'].values, 'g^--', lw=1.5,
              markersize=6, label='F1-macro', alpha=0.8)
    ax2p.set_ylim(0.60, 1.05)
    ax2p.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax2p.set_ylabel('F1-macro', color='green')
    ax2p.legend(loc='lower right', fontsize=8)

plt.suptitle('Biological Sequence Modeling: Paradigm Comparison', fontsize=12, y=1.01)
plt.tight_layout()
out_png = FIGURES / 'paradigm_comparison.png'
fig.savefig(out_png, dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', out_png)


Saved: /home/dpratapa/Capstone/reports/figures/final/paradigm_comparison.png


## 8) Full ranked tables

In [8]:
print('=== DNA: All models ranked by ROC-AUC ===')
dna_cols = [c for c in ['paradigm','model','accuracy','f1','roc_auc','pr_auc']
            if c in dna_all.columns]
print(dna_all[dna_cols].sort_values('roc_auc', ascending=False).to_string(index=False))

print('\n=== Protein: All models ranked by accuracy ===')
prot_cols = [c for c in ['paradigm','model','accuracy','f1_macro']
             if c in protein_all.columns]
print(protein_all[prot_cols].sort_values('accuracy', ascending=False).to_string(index=False))


=== DNA: All models ranked by ROC-AUC ===
 paradigm                 model  accuracy       f1  roc_auc   pr_auc
  NT-500M               xgboost   0.76875 0.767296 0.850700 0.854365
DNABERT-2               xgboost   0.75625 0.755332 0.846250 0.853206
  NT-500M         random_forest   0.75875 0.755387 0.846050 0.848444
DNABERT-2         random_forest   0.75250 0.740157 0.842894 0.842107
 Baseline         random_forest   0.74750 0.743003 0.837384 0.847125
   Hybrid          cnn+features   0.69750 0.740343 0.831688 0.846602
      CNN               seq_cnn   0.75000 0.748744 0.826375 0.837063
 Baseline            grad_boost   0.72625 0.715953 0.825606 0.838261
 Baseline               xgboost   0.73875 0.735108 0.824112 0.836746
 Baseline                logreg   0.73000 0.723785 0.823444 0.833903
 Baseline linear_svm_calibrated   0.73375 0.730038 0.822325 0.831796
DNABERT-2                logreg   0.71625 0.707097 0.785731 0.790491
DNABERT-2 linear_svm_calibrated   0.69125 0.686150 0.770131 0

## 9) Save final summary JSON

In [9]:
def safe(v):
    """Convert numpy scalars to Python native for JSON serialisation."""
    if pd.isna(v): return None
    if hasattr(v, 'item'): return v.item()
    return v

summary = {
    'dna': {
        'task': 'promoter vs non-promoter (binary, 200 bp, human GRCh38)',
        'n_sequences': 4000,
        'best_model': {
            'paradigm': str(dna_best.loc[dna_best['roc_auc'].idxmax(), 'paradigm']),
            'model':    str(dna_best.loc[dna_best['roc_auc'].idxmax(), 'model']),
            'roc_auc':  safe(dna_best['roc_auc'].max()),
            'accuracy': safe(dna_best.loc[dna_best['roc_auc'].idxmax(), 'accuracy']),
        },
        'paradigm_summary': [
            {k: safe(v) for k, v in row.items()}
            for row in dna_best[dna_cols].to_dict(orient='records')
        ],
    },
    'protein': {
        'task': 'Pfam family classification (10-class, human UniProt)',
        'n_sequences': 2293,
        'best_model': {
            'paradigm': str(protein_best.loc[protein_best['accuracy'].idxmax(), 'paradigm']),
            'model':    str(protein_best.loc[protein_best['accuracy'].idxmax(), 'model']),
            'accuracy': safe(protein_best['accuracy'].max()),
            'f1_macro': safe(protein_best.loc[protein_best['accuracy'].idxmax(), 'f1_macro']),
        },
        'paradigm_summary': [
            {k: safe(v) for k, v in row.items()}
            for row in protein_best[prot_cols].to_dict(orient='records')
        ],
    },
    'timestamp': pd.Timestamp.now().isoformat(),
}

out_json = REPORTS / 'final_summary.json'
with open(out_json, 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved:', out_json)
print(json.dumps(summary, indent=2))


Saved: /home/dpratapa/Capstone/reports/final_summary.json
{
  "dna": {
    "task": "promoter vs non-promoter (binary, 200 bp, human GRCh38)",
    "n_sequences": 4000,
    "best_model": {
      "paradigm": "NT-500M",
      "model": "xgboost",
      "roc_auc": 0.8507,
      "accuracy": 0.76875
    },
    "paradigm_summary": [
      {
        "paradigm": "Baseline",
        "model": "random_forest",
        "accuracy": 0.7475,
        "f1": 0.7430025445292621,
        "roc_auc": 0.8373843750000001,
        "pr_auc": 0.8471250524056608
      },
      {
        "paradigm": "CNN",
        "model": "seq_cnn",
        "accuracy": 0.75,
        "f1": 0.7487437185929648,
        "roc_auc": 0.826375,
        "pr_auc": 0.8370632258305786
      },
      {
        "paradigm": "Hybrid",
        "model": "cnn+features",
        "accuracy": 0.6975,
        "f1": 0.740343347639485,
        "roc_auc": 0.8316875000000001,
        "pr_auc": 0.8466018777391506
      },
      {
        "paradigm": "DNABERT-2

## 10) Key findings

### DNA task (promoter vs non-promoter)
- Transformer embeddings (NT-500M, DNABERT-2) improve over baselines but margins
  are modest -- engineered k-mer features already capture substantial signal.
- Tree-based classifiers (XGBoost, RF) extract the most from transformer embeddings;
  linear models underperform baselines in high-dimensional embedding spaces.
- The CNN alone is competitive with engineered baselines, suggesting raw sequence
  patterns are learnable without explicit feature construction.

### Protein task (Pfam family classification)
- Transformer embeddings (ProtBERT, ESM-2) dramatically outperform all other
  paradigms -- protein family structure is highly separable in pretrained
  embedding spaces.
- Linear models are strongest on protein embeddings (opposite of DNA), reflecting
  the clean linear separability of family-level structure in the embedding space.
- The gap between transformer and non-transformer paradigms is much larger for
  protein (~99% vs ~75-85%) than for DNA (~85% vs ~82-84%).

### General observation
- For regulatory DNA classification, the signal is diffuse and contextual;
  pretraining on sequence alone may not fully capture promoter-specific biology.
- For protein family classification, the evolutionary constraints encoded in
  pretrained protein LMs align directly with the task, yielding near-perfect results.